In [2]:

import os
import json
import torch
import matplotlib.pyplot as plt


# ============================================================
# CONFIGURATION
# ============================================================

RESULTS_ROOT = "./results"

DATASETS = [
    "THEIA_E3",
    "THEIA_E5",
    "TRACE_E3",
    "TRACE_E5",
]

METHODS = [
    "Flash",
    "Velox",
    "Orthrus",
]



TOP3_dic = {
    "THEIA_E3": 
    {
    
            "percent": 64
    },
    "THEIA_E5":{
    
            "percent": 90
    },
    "TRACE_E3":{
    
            "percent": 71
    },

 "TRACE_E5":{
    
            "percent": 77
    },

}

In [ ]:





# ============================================================
# METRIC CALCULATION
# ============================================================

def calculate_metrics(results_path):

    data = torch.load(
        results_path,
        map_location="cpu"
    )

    tp = set()
    fp = set()
    tn = set()
    fn = set()

    for key, item in data.items():

        y_hat = item["y_hat"]
        y_true = item["y_true"]

        if y_hat == 1:

            if y_true == 1:
                tp.add(key)
            else:
                fp.add(key)

        else:

            if y_true == 1:
                fn.add(key)
            else:
                tn.add(key)

    tp = len(tp)
    fp = len(fp)
    tn = len(tn)
    fn = len(fn)

    eps = 1e-10

    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)

    f1 = (
        2 * precision * recall /
        (precision + recall + eps)
    )

    return {
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "F1": f1,
    }


# ============================================================
# FIND RESULT FILE
# ============================================================

def find_result_file(dataset, method, configuration):

    path = os.path.join(
        RESULTS_ROOT,
        dataset,
        method,
        configuration,
        "results",
        "results.pth"
    )

    if os.path.exists(path):
        return path

    return None


# ============================================================
# DETERMINE CONFIGURATION
# ============================================================

def get_configurations():

    return [
        "base",
        "balanced",
    ]




# ============================================================
# COLLECT ALL RESULTS
# ============================================================

def collect_results():

    output = {}

    for dataset in DATASETS:

        output[dataset] = {}

        for method in METHODS:

            output[dataset][method]= {}
            output[dataset][method]["profiles"] =[]


            result_path1 = find_result_file(
                    dataset,
                    method,
                    dataset.lower()
                )
            result_path2 = find_result_file(
                    dataset,
                    method,
                    dataset.lower()+"_balanced"
                )

            if result_path1 is None :
                    print(
                        "MISSING:",
                        dataset,
                        method,
                    )
                    continue
            if result_path2 is None :
                    print(
                        "MISSING:",
                        dataset,
                        method,
                    )
            print(result_path1)
            metrics1 = calculate_metrics(
                    result_path1
                )
            metrics2 = calculate_metrics(result_path2)


            profile = {
                    
                    "removes_pct" : TOP3_dic[dataset],
                    "base" :{
                        "tp" : metrics1["TP"],
                        "fp" :metrics1["FP"],
                        "f1" :metrics1["F1"],
                    },
                    "balanced" :{
                        "tp" :metrics2["TP"],
                        "fp" :metrics2["FP"],
                        "f1" :metrics2["F1"],
                    },
                }
            output[dataset][method]= profile
    return output


# ============================================================
# SAVE JSON
# ============================================================

results = collect_results()

with open(
    "balanced_composition_results.json",
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=4
    )

print(
    "\nSaved:",
    "balanced_composition_results.json"
)


./results\THEIA_E3\Flash\theia_e3\results\results.pth
./results\THEIA_E3\Velox\theia_e3\results\results.pth
./results\THEIA_E3\Orthrus\theia_e3\results\results.pth
./results\THEIA_E5\Flash\theia_e5\results\results.pth
./results\THEIA_E5\Velox\theia_e5\results\results.pth
./results\THEIA_E5\Orthrus\theia_e5\results\results.pth
./results\TRACE_E3\Flash\trace_e3\results\results.pth
./results\TRACE_E3\Velox\trace_e3\results\results.pth
./results\TRACE_E3\Orthrus\trace_e3\results\results.pth
./results\TRACE_E5\Flash\trace_e5\results\results.pth
./results\TRACE_E5\Velox\trace_e5\results\results.pth
./results\TRACE_E5\Orthrus\trace_e5\results\results.pth

Saved: balanced_composition_results.json


In [7]:
import json


INPUT_FILE = "balanced_composition_results.json"
OUTPUT_FILE = "rq2_balanced_table.tex"


# ---------------------------------------------------------
# Formatting
# ---------------------------------------------------------

def fmt_int(x):
    if x is None:
        return "--"
    return f"{int(x):,}"


def fmt_pct(x):
    if x is None:
        return "--"
    return f"{x:.2f}\\%"


# ---------------------------------------------------------
# Load results
# ---------------------------------------------------------

with open(INPUT_FILE, "r") as f:
    results = json.load(f)


# ---------------------------------------------------------
# Build LaTeX
# ---------------------------------------------------------

lines = []

lines.append(r"\begin{table*}[t]")
lines.append(r"\centering")

lines.append(
    r"\caption{Detection performance before and after median-based behavioral"
 r"rebalancing. The reported percentage indicates the fraction of training nodes"
 r"removed by the rebalancing procedure. TP and FP denote true positives and"
 r"false positives, respectively, and F1 is reported as a percentage.}"
)
lines.append(r"\label{tab:balancing}")

lines.append(r"\small")
lines.append(r"\begin{tabular}{ll r rrr rrr}")
lines.append(r"\hline")

lines.append(
    r"\textbf{Dataset} & "
    r"\textbf{Removed} & "
    r"\textbf{PIDS} & "
    r"\multicolumn{3}{c}{\textbf{Base}} & "
    r"\multicolumn{3}{c}{\textbf{Balanced}} \\"
)

lines.append(
    r"\cline{4-6} \cline{7-9}"
)

lines.append(
    r" & & & "
    r"\textbf{TP} & \textbf{FP} & \textbf{F1} &  "
    r"\textbf{TP} & \textbf{FP} &\textbf{F1}  \\"
)

lines.append(r"\hline")


# ---------------------------------------------------------
# Rows
# ---------------------------------------------------------

for dataset in DATASETS:

    dataset_data = results.get(dataset, {})
    count = 0  
    for pid in METHODS:

        pid_data = dataset_data.get(pid, {})

        removed_pct = pid_data.get("removes_pct", [])["percent"]

        count +=1
        base_tp = pid_data["base"]["tp"]
        base_fp = pid_data["base"]["fp"]
        base_f1 = f"{pid_data['base']['f1'] * 100:.2f}\%"


        random_tp = pid_data["balanced"]["tp"]
        random_fp = pid_data["balanced"]["fp"]
        random_f1 = f"{pid_data['balanced']['f1'] * 100:.2f}\%"

            # Avoid repeating dataset / PIDS names
        if count == 2 :
                dataset_cell = dataset.replace("_", "-")
                removed_pct = fmt_pct(removed_pct)
        else:
                dataset_cell = ""
                removed_pct = ""

        lines.append(
                f"{dataset_cell} & "
                f"{removed_pct} & "
                f"{pid} & "
                f"{fmt_int(base_tp)} & "
                f"{fmt_int(base_fp)} & "
                f"{base_f1} & "
                f"{fmt_int(random_tp)} & "
                f"{fmt_int(random_fp)} &"
                f"{random_f1} \\\\"
            )
        lines.append( r"\cline{3-9} ")
    lines.append(r"\hline")
    lines.append(r"\hline")


lines.append(r"\end{tabular}")
lines.append(r"\end{table*}")


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

with open(OUTPUT_FILE, "w") as f:
    f.write("\n".join(lines))


print(f"Saved: {OUTPUT_FILE}")

Saved: rq2_balanced_table.tex


<>:91: SyntaxWarning: invalid escape sequence '\%'
<>:96: SyntaxWarning: invalid escape sequence '\%'
<>:91: SyntaxWarning: invalid escape sequence '\%'
<>:96: SyntaxWarning: invalid escape sequence '\%'
C:\Users\Asus\AppData\Local\Temp\ipykernel_16068\1577588077.py:91: SyntaxWarning: invalid escape sequence '\%'
  base_f1 = f"{pid_data['base']['f1'] * 100:.2f}\%"
C:\Users\Asus\AppData\Local\Temp\ipykernel_16068\1577588077.py:96: SyntaxWarning: invalid escape sequence '\%'
  random_f1 = f"{pid_data['balanced']['f1'] * 100:.2f}\%"
